[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb5_train_joint_seeds.ipynb)

# nb5: joint model, seeds 1 and 2 (optional)
Trains `joint_s1` and `joint_s2` exactly as `joint_s0` (only `--seed` changes: init, batch order and noise; the training set is simulation seed 0 for all). Each run skips if finished and resumes if interrupted. Then writes the seed report and figure (new files only). **Run only if GPU time is available.**

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. Check nb0 outputs

In [ ]:
import os
for f in ["cache/pendulum_T64_dt0.04_train.npz", "cache/pendulum_T64_dt0.04_heldout.npz", "cache/video_metric.json"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"missing {f}: run nb0_benchmark first"
print("nb0 outputs present")

## 6. Train `joint_s1`

In [ ]:
%%bash
set -o pipefail
mkdir -p "/content/drive/MyDrive/physgen_cvpr27/joint_s1"
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant joint --run_name joint_s1 --seed 1 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/joint_s1/console_$(date +%s).log"

## 7. Train `joint_s2`

In [ ]:
%%bash
set -o pipefail
mkdir -p "/content/drive/MyDrive/physgen_cvpr27/joint_s2"
cd /content/repo
PYTHONUNBUFFERED=1 python -u -m cvpr27.train --variant joint --run_name joint_s2 --seed 2 2>&1 | tee -a "/content/drive/MyDrive/physgen_cvpr27/joint_s2/console_$(date +%s).log"

## 8. Seed report (joint_s0 / s1 / s2, read from their files)

In [ ]:
!cd /content/repo && python -u -m cvpr27.seeds --tag seeds

In [ ]:
from IPython.display import Image, display
display(Image(f"{ROOT}/eval_seeds/fig_seeds_training.png"))